In [ ]:
!pip install pyyaml

In [ ]:
import sys
import os

# Add the parent directory (src) to the Python path
sys.path.append(os.path.dirname(os.getcwd()))

In [0]:
from utility.catalog_utils import CatalogUtils
import yaml

In [ ]:
current_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

# Get the directory containing the notebook (remove the notebook filename)
src_directory = '/'.join(current_notebook_path.split('/')[:-1])

In [ ]:
# Build the CSV file path
file_path_parameter = f"/Workspace{src_directory}/util/parameters.yml"

with open(file_path_parameter, "r") as file:
    config = yaml.safe_load(file)

In [ ]:
# config table variable
catalog_ = config["config_catalog"]
schema_ = config["config_schema"]
table_nm = config["config_table"]

In [0]:
CatalogUtils.ensure_catalog_and_schema(spark, catalog_, schema_)

Catalog 'migration_catalog' already exists.
Created schema: migration_catalog.logging


In [0]:
# Build the CSV file path
file_path = f"/Workspace{src_directory}/util/config.csv"
print(f"CSV file path: {file_path}")

CSV file path: /Workspace/Shared/AssetBundleSimulation_V0/my_sql_mig/testing/config.csv


In [0]:
from pyspark.sql.functions import col, to_timestamp
import pandas as pd

df = pd.read_csv(file_path)
spark_df = spark.createDataFrame(df)

last_column = spark_df.columns[-1]
spark_df = spark_df.withColumn(last_column, to_timestamp(col(last_column)))

table_exists = any(row.tableName == table_nm for row in spark.sql(f"SHOW TABLES IN {catalog_}.{schema_}").collect())

if not table_exists:
    spark_df.write.mode("overwrite").saveAsTable(f"{catalog_}.{schema_}.{table_nm}")
    print(f"Table {catalog_}.{schema_}.{table_nm} created.")
else:
    spark_df.createOrReplaceTempView("temp_table")

    spark.sql(f"""
              merge into {catalog_}.{schema_}.{table_nm} t
              using temp_table s
              on t.Target_Table_Catalog = s.Target_Table_Catalog and t.Target_table_schema = s.Target_table_schema and t.Target_table_name = s.Target_table_name
              when not matched then insert *
              """)
    print(f"Table {catalog_}.{schema_}.{table_nm} updated.")

Table migration_catalog.logging.config_table created.
